In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/clean-text-module/emoji-1.7.0/emoji-1.7.0/README.rst
/kaggle/input/clean-text-module/emoji-1.7.0/emoji-1.7.0/CHANGES.md
/kaggle/input/clean-text-module/emoji-1.7.0/emoji-1.7.0/setup.cfg
/kaggle/input/clean-text-module/emoji-1.7.0/emoji-1.7.0/LICENSE.txt
/kaggle/input/clean-text-module/emoji-1.7.0/emoji-1.7.0/MANIFEST.in
/kaggle/input/clean-text-module/emoji-1.7.0/emoji-1.7.0/PKG-INFO
/kaggle/input/clean-text-module/emoji-1.7.0/emoji-1.7.0/setup.py
/kaggle/input/clean-text-module/emoji-1.7.0/emoji-1.7.0/tests/test_dict.py
/kaggle/input/clean-text-module/emoji-1.7.0/emoji-1.7.0/tests/test_versions.py
/kaggle/input/clean-text-module/emoji-1.7.0/emoji-1.7.0/tests/test_core.py
/kaggle/input/clean-text-module/emoji-1.7.0/emoji-1.7.0/tests/__init__.py
/kaggle/input/clean-text-module/emoji-1.7.0/emoji-1.7.0/tests/test_deprecation.py
/kaggle/input/clean-text-module/emoji-1.7.0/emoji-1.7.0/tests/test_unicode_codes.py
/kaggle/input/clean-text-module/emoji-1.7.0/emoji-1.7.0/emoji/cor

In [2]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
#pip install torch transformers datasets peft optuna scikit-learn tqdm cleantext

In [4]:
#pip install transformers datasets peft accelerate optuna scikit-learn tensorboard


In [5]:
#pip install ftfy evaluate clean-text==0.6.0

In [6]:
'''

import os
import math
import evaluate
import numpy as np
from datasets import Dataset
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training  # if using int8
import optuna
from datetime import datetime

set_seed(42)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

'''

'\n\nimport os\nimport math\nimport evaluate\nimport numpy as np\nfrom datasets import Dataset\nfrom sklearn.metrics import roc_auc_score\nfrom sklearn.model_selection import StratifiedKFold\nfrom transformers import (\n    AutoTokenizer,\n    AutoModelForSequenceClassification,\n    TrainingArguments,\n    Trainer,\n    DataCollatorWithPadding,\n    set_seed,\n)\nfrom peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training  # if using int8\nimport optuna\nfrom datetime import datetime\n\nset_seed(42)\nos.environ["TOKENIZERS_PARALLELISM"] = "false"\n\n'

# Let's delve into the datasets !

In [7]:
train = pd.read_csv('/kaggle/input/jigsaw-agile-community-rules/train.csv')
train.head()

,row_id,body,rule,subreddit,positive_example_1,positive_example_2,negative_example_1,negative_example_2,rule_violation
0,0,Banks don't want you to know this! Click here ...,"No Advertising: Spam, referral links, unsolici...",Futurology,If you could tell your younger self something ...,hunt for lady for jack off in neighbourhood ht...,Watch Golden Globe Awards 2017 Live Online in ...,"DOUBLE CEE x BANDS EPPS - ""BIRDS""\n\nDOWNLOAD/...",0
1,1,SD Stream [ ENG Link 1] (http://www.sportsstre...,"No Advertising: Spam, referral links, unsolici...",soccerstreams,[I wanna kiss you all over! Stunning!](http://...,LOLGA.COM is One of the First Professional Onl...,#Rapper \n🚨Straight Outta Cross Keys SC 🚨YouTu...,[15 Amazing Hidden Features Of Google Search Y...,0
2,2,Lol. Try appealing the ban and say you won't d...,No legal advice: Do not offer or request legal...,pcmasterrace,Don't break up with him or call the cops. If ...,It'll be dismissed: https://en.wikipedia.org/w...,Where is there a site that still works where y...,Because this statement of his is true. It isn'...,1
3,3,she will come your home open her legs with an...,"No Advertising: Spam, referral links, unsolici...",sex,Selling Tyrande codes for 3€ to paypal. PM. \n...,tight pussy watch for your cock get her at thi...,NSFW(obviously) http://spankbang.com/iy3u/vide...,Good News ::Download WhatsApp 2.16.230 APK for...,1
4,4,code free tyrande --->>> [Imgur](http://i.imgu...,"No Advertising: Spam, referral links, unsolici...",hearthstone,wow!! amazing reminds me of the old days.Well...,seek for lady for sex in around http://p77.pl/...,must be watch movie https://sites.google.com/s...,We're streaming Pokemon Veitnamese Crystal RIG...,1


Looking at first at the header we can see make some observations:

* 'body' is the text and comment we are looking at to see if it made rule violations and we can try to check is there are any duplicates
  
* 'rule' is what the comment is juged on so we can maybe try to see if the text contents have words that are related to 'rule', btw we can make categories out of it so looking into its values may be interesting

* 'subreddit' is the forum where it was posted so we can try to look if the 'body' content is related to the theme of the 'subreddit'

* 'positive_examples_n' are comments that violates the rule so we can try to see if the 'body' context has similarity with it and the theme of the content

* 'negative_examples_n' are comments that doesnt violate rules and we can try to see the similarities, the theme of the content and then how different it is from 'postive_examples_n'

* 'rule_violation' indicates if the comment is violating the rule, it is what we will try to predict. 0 if its not and 1 if it is violating rule


In [8]:
test = pd.read_csv('/kaggle/input/jigsaw-agile-community-rules/test.csv')
test.head()

,row_id,body,rule,subreddit,positive_example_1,positive_example_2,negative_example_1,negative_example_2
0,2029,NEW RAP GROUP 17. CHECK US OUT https://soundcl...,"No Advertising: Spam, referral links, unsolici...",hiphopheads,"Hey, guys, just wanted to drop in and invite y...",Cum Swallowing Hottie Katrina Kaif Cartoon Xvi...,SD Stream Eng - [Chelsea TV USA](http://soccer...,HD Streams: |[ENG HD Stoke vs Manchester Unite...
1,2030,Make your life comfortable. Get up to 15% Disc...,No legal advice: Do not offer or request legal...,AskReddit,Get a lawyer and get the security camera foota...,That isn't drastic. You tried reaching out to ...,So what are you going to do with the insurance...,It's just for Austria & Germany. If you still ...
2,2031,Kickin' ass and selling underwear!\nJust made ...,"No Advertising: Spam, referral links, unsolici...",gonewild,Good story my friend. Check out my blog at ht...,If you know what exactly you need then you don...,CENTIPEDES\n\nSOME BASED PATRIOTS HAVE CREATED...,[So great! Thanks for sharing.](http://www.che...
3,2032,watch hooters best therein http://clickan...,"No Advertising: Spam, referral links, unsolici...",personalfinance,"Earn 50,000 bonus points with Chase Sapphire P...","Cool, front page! I made this print along with...",[Full HD Movie Online Free](http://www.flickma...,* Karambit Black Pearl\n* 0.02137822 Float (un...
4,2033,bitches for free at this point show all h...,"No Advertising: Spam, referral links, unsolici...",Showerthoughts,code free tyrande --->>> [Imgur](http://i.imgu...,My trade link\nhttps://steamcommunity.com/trad...,**HD** [ mio Stadium 102 HD](http://www.genti....,Infographics is an incredible method for showi...


* We can see here that the only missing column is 'rule_violation'

In [9]:
sample_submission = pd.read_csv('/kaggle/input/jigsaw-agile-community-rules/sample_submission.csv')
sample_submission.head()

,row_id,rule_violation
0,2029,0.5
1,2030,0.5
2,2031,0.5
3,2032,0.5
4,2033,0.5


* We can see that we will have to return prediction probabilities of if it was rule violation

# Let's sanitize texts 

In [10]:
import re

def replace_urls(text):
    # Pattern to match common URLs
    url_pattern = r'https?://\S+'
    return re.sub(url_pattern, '[URL]', text)

In [11]:
train

,row_id,body,rule,subreddit,positive_example_1,positive_example_2,negative_example_1,negative_example_2,rule_violation
0,0,Banks don't want you to know this! Click here ...,"No Advertising: Spam, referral links, unsolici...",Futurology,If you could tell your younger self something ...,hunt for lady for jack off in neighbourhood ht...,Watch Golden Globe Awards 2017 Live Online in ...,"DOUBLE CEE x BANDS EPPS - ""BIRDS""\n\nDOWNLOAD/...",0
1,1,SD Stream [ ENG Link 1] (http://www.sportsstre...,"No Advertising: Spam, referral links, unsolici...",soccerstreams,[I wanna kiss you all over! Stunning!](http://...,LOLGA.COM is One of the First Professional Onl...,#Rapper \n🚨Straight Outta Cross Keys SC 🚨YouTu...,[15 Amazing Hidden Features Of Google Search Y...,0
2,2,Lol. Try appealing the ban and say you won't d...,No legal advice: Do not offer or request legal...,pcmasterrace,Don't break up with him or call the cops. If ...,It'll be dismissed: https://en.wikipedia.org/w...,Where is there a site that still works where y...,Because this statement of his is true. It isn'...,1
3,3,she will come your home open her legs with an...,"No Advertising: Spam, referral links, unsolici...",sex,Selling Tyrande codes for 3€ to paypal. PM. \n...,tight pussy watch for your cock get her at thi...,NSFW(obviously) http://spankbang.com/iy3u/vide...,Good News ::Download WhatsApp 2.16.230 APK for...,1
4,4,code free tyrande --->>> [Imgur](http://i.imgu...,"No Advertising: Spam, referral links, unsolici...",hearthstone,wow!! amazing reminds me of the old days.Well...,seek for lady for sex in around http://p77.pl/...,must be watch movie https://sites.google.com/s...,We're streaming Pokemon Veitnamese Crystal RIG...,1
...,...,...,...,...,...,...,...,...,...
2024,2024,Please edit your post so it is readable. These...,No legal advice: Do not offer or request legal...,relationships,"I'm not ok with this in anyway, and think you ...",See a lawyer under the guise that you want thi...,"This is just untrue. OP is 13, not an adult -...",Why should I care about all the bicyclists I i...,1
2025,2025,"Yes, and in a right to work state they can eve...",No legal advice: Do not offer or request legal...,legaladvice,Move as much of your assets as you can offshor...,We have great consumer protection laws. There'...,"LPT piratebay, transmission, vpn. Get the musi...","It's not so much that I killed them, it's that...",0
2026,2026,**HD** Streams: |ENG **HD**[ Watch here..PC & ...,"No Advertising: Spam, referral links, unsolici...",soccerstreams,stitopdisca1987.tumblr.com - sex Take girl for...,this girl get sex going to to old http://mrk....,NO ADS | NO ADS | NO ADS\n\nWe show all SOCCER...,[So great! Thanks for sharing.](http://www.che...,1
2027,2027,No. Not when doing so obviously presents a saf...,No legal advice: Do not offer or request legal...,politics,SHE ISNT A BIRTHING CHAMBER BUT EQUALLY THE BA...,"Jail? What are you, ten years old? If they pro...",Who cares about that when I can keep raping in...,send me a private message; I may be able to he...,1


In [12]:
print(replace_urls(train.iloc[0].negative_example_2))

DOUBLE CEE x BANDS EPPS - "BIRDS"

DOWNLOAD/STREAM:

[URL]


In [13]:
# Decode and Encode into UTF-8

def sanitize_text(text):
    """
    Ensures the text is valid UTF-8 by encoding and decoding it.
    Any invalid characters will be replaced with �.
    """
    if not isinstance(text, str):
        text = str(text)
    
    # Encode to UTF-8 bytes, replacing invalid sequences
    encoded = text.encode("utf-8", errors="replace")
    
    # Decode back to string
    return encoded.decode("utf-8", errors="replace")


In [14]:
# Apply both previous function to sanitize text

def clean_text(text):
    """Apply URL replacement and UTF-8 sanitization to text."""
    return sanitize_text(replace_urls(text))

In [15]:
import re
import unicodedata

def cleaner(text):
    """
    Replicates the clean_text.clean function with the same parameters.
    """
    if not isinstance(text, str):
        return text

    # Fix unicode (normalize to NFC form)
    if True:  # fix_unicode=True
        text = unicodedata.normalize('NFC', text)
    
    # Convert to ASCII (remove non-ASCII characters)
    if True:  # to_ascii=True
        text = text.encode('ascii', 'ignore').decode('ascii')
    
    # Handle URLs
    if True:  # no_urls=True
        url_pattern = r'https?://\S+|www\.\S+'
        text = re.sub(url_pattern, '<URL>', text)
    
    # Handle emails
    if True:  # no_emails=True
        email_pattern = r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b'
        text = re.sub(email_pattern, '<EMAIL>', text)
    
    # Handle phone numbers (basic pattern)
    if True:  # no_phone_numbers=True
        phone_pattern = r'\b(?:\+\d{1,2}\s?)?\(?\d{3}\)?[\s.-]?\d{3}[\s.-]?\d{4}\b'
        text = re.sub(phone_pattern, '<PHONE>', text)
    
    # Note: The original clean_text keeps these False, so we don't modify:
    # - lower=False (don't lowercase)
    # - no_line_breaks=False (keep line breaks)
    # - no_numbers=False (keep numbers)
    # - no_digits=False (keep digits)
    # - no_currency_symbols=False (keep currency symbols)
    # - no_punct=False (keep punctuation)
    
    return text

In [16]:
# Let's make a function that builds a prompt out of a row in a dataframe

def build_prompt(row):
    base_prompt = """You are an expert content moderator. Your task is to determine if a given Reddit comment violates a specified subreddit rule. 
Carefully analyze the comment in the context of the rule.

- Only answer "Yes" if the comment clearly breaks the rule.
- Answer "No" if the comment complies or does not violate the rule.
- Do not provide any explanation or additional text; respond strictly with "Yes" or "No".
- Consider the examples carefully before making your judgment.

Classify the comment below:"""

    prompt = f"""
{base_prompt}

Subreddit: {row['subreddit']}

Rule: {row['rule']}

Examples:

1) {cleaner(row['positive_example_1'])}
Answer: Yes

2) {cleaner(row['positive_example_2'])}
Answer: Yes

3) {cleaner(row['negative_example_1'])}
Answer: No

4) {cleaner(row['negative_example_2'])}
Answer: No

----

Comment: {cleaner(row['body'])}
Answer:
"""
    return prompt

In [17]:
row = train.iloc[1]
for col, val in row.items():
    print(f"{col}: {val}\n")

row_id: 1

body: SD Stream [ ENG Link 1] (http://www.sportsstreams247.com/astra-giurgiu-vs-fc-austria-wien/)

rule: No Advertising: Spam, referral links, unsolicited advertising, and promotional content are not allowed.

subreddit: soccerstreams

positive_example_1: [I wanna kiss you all over! Stunning!](http://www.oilflush.life/2017/01/26/6/)

positive_example_2: LOLGA.COM is One of the First Professional Online Gold sites. By Now, As A Game Gold Seller, we've over more than 5 yrs Of Experience And Can Pass That On To Our Customers.

negative_example_1: #Rapper 
🚨Straight Outta Cross Keys SC 🚨YouTube Search Beanie 864 Click Link BELOW To Hear Hit Single
  "Ah Man" 
 Beanie 864 FEAT King Kota 
 (King Kota Is Only 15!) Lit 🌡🔥👍💵💯Fr Fr 
https://youtu.be/tLqbV1Jmt5Y

negative_example_2: [15 Amazing Hidden Features Of Google Search You Probably Don’t Know](http://www.madpeoples.com/2017/01/02  No one would argue the fact that Google is one of the most useful sihttp://www.madpeoples.com/2016

In [18]:
print(build_prompt(train.iloc[1]))


You are an expert content moderator. Your task is to determine if a given Reddit comment violates a specified subreddit rule. 
Carefully analyze the comment in the context of the rule.

- Only answer "Yes" if the comment clearly breaks the rule.
- Answer "No" if the comment complies or does not violate the rule.
- Do not provide any explanation or additional text; respond strictly with "Yes" or "No".
- Consider the examples carefully before making your judgment.

Classify the comment below:

Subreddit: soccerstreams

Rule: No Advertising: Spam, referral links, unsolicited advertising, and promotional content are not allowed.

Examples:

1) [I wanna kiss you all over! Stunning!](<URL>
Answer: Yes

2) LOLGA.COM is One of the First Professional Online Gold sites. By Now, As A Game Gold Seller, we've over more than 5 yrs Of Experience And Can Pass That On To Our Customers.
Answer: Yes

3) #Rapper 
Straight Outta Cross Keys SC YouTube Search Beanie 864 Click Link BELOW To Hear Hit Single
 

In [19]:
print(len(build_prompt(train.iloc[1])))

1429


# Let's try at first to use classification model using sk learn and a tokenizer

In [20]:
# Let's create a function that create the dataset 

def create_dataset(df: pd.DataFrame) -> pd.DataFrame:
    
    df = df.copy()

    # Apply build_prompt row by row
    prompts = df.apply(build_prompt, axis=1)

    # Build new dataframe
    new_df = pd.DataFrame({
        "prompts": prompts,
        "rule_violation": df["rule_violation"]
    })

    return new_df

In [21]:
train_dataset = create_dataset(train)

In [22]:
train_dataset.head()

,prompts,rule_violation
0,\nYou are an expert content moderator. Your ta...,0
1,\nYou are an expert content moderator. Your ta...,0
2,\nYou are an expert content moderator. Your ta...,1
3,\nYou are an expert content moderator. Your ta...,1
4,\nYou are an expert content moderator. Your ta...,1


In [23]:
train_dataset.rule_violation.value_counts()

rule_violation
1    1031
0     998
Name: count, dtype: int64

In [24]:
'''
from sklearn.linear_model import LogisticRegression, Perceptron, RidgeClassifier, SGDClassifier, PassiveAggressiveClassifier
from sklearn.neighbors import KNeighborsClassifier, NearestCentroid, RadiusNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB, ComplementNB, CategoricalNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier, BaggingClassifier, 
    AdaBoostClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier,
    StackingClassifier, VotingClassifier
)
from sklearn.neural_network import MLPClassifier

# Dictionary of classifiers grouped by category
classifiers = {
    "Linear Models": [
        LogisticRegression,
        Perceptron,
        RidgeClassifier,
        SGDClassifier,
        PassiveAggressiveClassifier
    ],
    
    "Nearest Neighbors": [
        KNeighborsClassifier,
        NearestCentroid,
        RadiusNeighborsClassifier
    ],
    
    "Naive Bayes": [
        GaussianNB,
        MultinomialNB,
        BernoulliNB,
        ComplementNB,
        CategoricalNB
    ],
    
    "Decision Trees & Ensembles": [
        DecisionTreeClassifier,
        RandomForestClassifier,
        ExtraTreesClassifier,
        BaggingClassifier,
        AdaBoostClassifier,
        GradientBoostingClassifier,
        HistGradientBoostingClassifier,
        StackingClassifier,
        VotingClassifier
    ],
    
    "Neural Network": [
        MLPClassifier
    ]
}
'''

'\nfrom sklearn.linear_model import LogisticRegression, Perceptron, RidgeClassifier, SGDClassifier, PassiveAggressiveClassifier\nfrom sklearn.neighbors import KNeighborsClassifier, NearestCentroid, RadiusNeighborsClassifier\nfrom sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB, ComplementNB, CategoricalNB\nfrom sklearn.tree import DecisionTreeClassifier\nfrom sklearn.ensemble import (\n    RandomForestClassifier, ExtraTreesClassifier, BaggingClassifier, \n    AdaBoostClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier,\n    StackingClassifier, VotingClassifier\n)\nfrom sklearn.neural_network import MLPClassifier\n\n# Dictionary of classifiers grouped by category\nclassifiers = {\n    "Linear Models": [\n        LogisticRegression,\n        Perceptron,\n        RidgeClassifier,\n        SGDClassifier,\n        PassiveAggressiveClassifier\n    ],\n    \n    "Nearest Neighbors": [\n        KNeighborsClassifier,\n        NearestCentroid,\n        Radius

In [25]:

import torch
from torch import Tensor
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained('/kaggle/input/qwen-3-embedding/transformers/0.6b/1', padding_side='left', torch_dtype=torch.float16)
model = AutoModel.from_pretrained('/kaggle/input/qwen-3-embedding/transformers/0.6b/1')

2025-09-16 11:31:09.787968: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758022269.972545      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758022270.026992      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [26]:

# 3. Convert texts -> embeddings
import time
from tqdm import tqdm

# Detect GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Move model to GPU
model = model.to(device)

def get_embeddings(texts, batch_size=32):
    all_embeddings = []
    n_texts = len(texts)

    for i in tqdm(range(0, n_texts, batch_size), desc="Embedding texts", unit="batch"):
        batch_texts = texts[i:i+batch_size]
        inputs = tokenizer(
            list(batch_texts),
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to(device)  # 🚀 send inputs to GPU

        with torch.no_grad():
            outputs = model(**inputs)
            embeddings = outputs.last_hidden_state.mean(dim=1)  

        all_embeddings.append(embeddings.cpu().numpy())  # move back to CPU for NumPy

    return np.vstack(all_embeddings)

# 3. Get embeddings for the DataFrame column
X = get_embeddings(train_dataset["prompts"])
y = train_dataset["rule_violation"].values


Using device: cuda


Embedding texts: 100%|██████████| 64/64 [02:40<00:00,  2.50s/batch]


In [27]:

from sklearn.model_selection import train_test_split

# 4. Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [28]:
X_train

array([[-5.8545041e-01, -6.2417728e-01, -9.3837762e-01, ...,
         2.0152626e+00,  9.5959532e-01,  4.7181219e-01],
       [ 9.8331606e-01, -4.2641559e+00, -7.3912233e-01, ...,
        -4.4257367e-01,  5.6012487e-01, -3.2425088e-01],
       [ 7.7360684e-01,  2.2167249e-03, -1.1338773e+00, ...,
         8.4730607e-01,  1.2284288e+00,  1.9453592e+00],
       ...,
       [ 4.7409061e-01, -5.1445780e+00, -7.8847611e-01, ...,
        -2.6073521e-01,  3.8099825e-01,  1.3125928e-01],
       [ 1.2967819e+00, -4.2594028e+00, -1.0566361e+00, ...,
         1.2370250e+00,  2.1636090e+00, -3.2216170e-01],
       [ 2.2267985e+00, -2.2662022e+00, -1.0876576e+00, ...,
         7.1739805e-01,  1.4078677e+00,  1.7196052e-01]], dtype=float32)

In [29]:
'''
# 5. Evaluate classifiers
import time
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression, SGDClassifier



results = {}

for category, models in classifiers.items():
    for model_class in models:
        try:
            start_time = time.time()

            # Some models need max_iter or special params
            if model_class == LogisticRegression:
                clf = model_class(max_iter=2000)
            elif model_class == SGDClassifier:
                clf = model_class(loss="log_loss", max_iter=2000)  # ensure probabilistic output
            else:
                clf = model_class()

            clf.fit(X_train, y_train)

            # Some classifiers don’t have predict_proba → fallback to decision_function
            if hasattr(clf, "predict_proba"):
                y_proba = clf.predict_proba(X_test)[:, 1]
            elif hasattr(clf, "decision_function"):
                y_proba = clf.decision_function(X_test)
            else:
                print(f"⏩ Skipped {model_class.__name__} (no probability/decision output)")
                continue

            auc = roc_auc_score(y_test, y_proba)
            elapsed = time.time() - start_time

            # Save result
            results[f"{category} - {model_class.__name__}"] = auc

            # Print live status
            print(f"✅ {category} - {model_class.__name__} | ROC AUC: {auc:.4f} | Time: {elapsed:.2f}s")

        except Exception as e:
            results[f"{category} - {model_class.__name__}"] = f"Error: {e}"
            print(f"❌ {category} - {model_class.__name__} failed: {e}")

# Final results table
results_clean = []
for model, auc in results.items():
    if isinstance(auc, (int, float)):
        results_clean.append([model, auc, None])  # no error
    else:
        results_clean.append([model, None, auc])  # store error separately

results_df = pd.DataFrame(results_clean, columns=["Model", "ROC AUC", "Error"])
results_df = results_df.sort_values(by="ROC AUC", ascending=False, na_position="last")
print(results_df)
'''

'\n# 5. Evaluate classifiers\nimport time\nimport pandas as pd\nfrom sklearn.metrics import roc_auc_score\nfrom sklearn.linear_model import LogisticRegression, SGDClassifier\n\n\n\nresults = {}\n\nfor category, models in classifiers.items():\n    for model_class in models:\n        try:\n            start_time = time.time()\n\n            # Some models need max_iter or special params\n            if model_class == LogisticRegression:\n                clf = model_class(max_iter=2000)\n            elif model_class == SGDClassifier:\n                clf = model_class(loss="log_loss", max_iter=2000)  # ensure probabilistic output\n            else:\n                clf = model_class()\n\n            clf.fit(X_train, y_train)\n\n            # Some classifiers don’t have predict_proba → fallback to decision_function\n            if hasattr(clf, "predict_proba"):\n                y_proba = clf.predict_proba(X_test)[:, 1]\n            elif hasattr(clf, "decision_function"):\n                y_

train test split 0.2
                                                
                                                Model   ROC AUC  \
4         Linear Models - PassiveAggressiveClassifier  0.740418   
21                     Neural Network - MLPClassifier  0.733276   
1                          Linear Models - Perceptron  0.718038   
3                       Linear Models - SGDClassifier  0.716326   
13  Decision Trees & Ensembles - RandomForestClass...  0.713917   
14  Decision Trees & Ensembles - ExtraTreesClassifier  0.713379   
0                  Linear Models - LogisticRegression  0.711801   
17  Decision Trees & Ensembles - GradientBoostingC...  0.698496   
18  Decision Trees & Ensembles - HistGradientBoost...  0.691501   
2                     Linear Models - RidgeClassifier  0.683747   
9                           Naive Bayes - BernoulliNB  0.662174   
15     Decision Trees & Ensembles - BaggingClassifier  0.658237   
5            Nearest Neighbors - KNeighborsClassifier  0.637862   
7                            Naive Bayes - GaussianNB  0.637092   
16    Decision Trees & Ensembles - AdaBoostClassifier  0.622441   
12  Decision Trees & Ensembles - DecisionTreeClass...  0.567531   
6       Nearest Neighbors - RadiusNeighborsClassifier       NaN   
8                         Naive Bayes - MultinomialNB       NaN   
10                         Naive Bayes - ComplementNB       NaN   
11                        Naive Bayes - CategoricalNB       NaN   
19    Decision Trees & Ensembles - StackingClassifier       NaN   
20      Decision Trees & Ensembles - VotingClassifier       NaN   

train test split 0.25

                                                Model   ROC AUC  \
1                          Linear Models - Perceptron  0.737584   
4         Linear Models - PassiveAggressiveClassifier  0.729499   
3                       Linear Models - SGDClassifier  0.717457   
0                  Linear Models - LogisticRegression  0.708266   
13  Decision Trees & Ensembles - RandomForestClass...  0.707347   
21                     Neural Network - MLPClassifier  0.705290   
14  Decision Trees & Ensembles - ExtraTreesClassifier  0.703935   
18  Decision Trees & Ensembles - HistGradientBoost...  0.698592   
15     Decision Trees & Ensembles - BaggingClassifier  0.687040   
17  Decision Trees & Ensembles - GradientBoostingC...  0.686425   
2                     Linear Models - RidgeClassifier  0.676128   
9                           Naive Bayes - BernoulliNB  0.665324   
7                            Naive Bayes - GaussianNB  0.650844   
16    Decision Trees & Ensembles - AdaBoostClassifier  0.644831   
5            Nearest Neighbors - KNeighborsClassifier  0.635905   
12  Decision Trees & Ensembles - DecisionTreeClass...  0.570227   
6       Nearest Neighbors - RadiusNeighborsClassifier       NaN   
8                         Naive Bayes - MultinomialNB       NaN   
10                         Naive Bayes - ComplementNB       NaN   
11                        Naive Bayes - CategoricalNB       NaN   
19    Decision Trees & Ensembles - StackingClassifier       NaN   
20      Decision Trees & Ensembles - VotingClassifier       NaN   

                                                Error  
1                                                None  
4                                                None  
3                                                None  
0                                                None  
13                                               None  
21                                               None  
14                                               None  
18                                               None  
15                                               None  
17                                               None  
2                                                None  
9                                                None  
7                                                None  
16                                               None  
5                                                None  
12                                               None  
6   Error: No neighbors found for test samples arr...  
8   Error: Negative values in data passed to Multi...  
10  Error: Negative values in data passed to Compl...  
11  Error: Negative values in data passed to Categ...  
19  Error: StackingClassifier.__init__() missing 1...  
20  Error: VotingClassifier.__init__() missing 1 r...  

In [30]:
'''
import optuna
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold


def objective(trial):
    # Suggest hyperparameters
    C = trial.suggest_loguniform("C", 1e-4, 1e2)   # regularization strength
    max_iter = trial.suggest_int("max_iter", 500, 5000)
    tol = trial.suggest_loguniform("tol", 1e-5, 1e-1)
    loss = trial.suggest_categorical("loss", ["hinge", "squared_hinge"])
    fit_intercept = trial.suggest_categorical("fit_intercept", [True, False])
    early_stopping = trial.suggest_categorical("early_stopping", [True, False])
    
    # Define classifier
    clf = PassiveAggressiveClassifier(
        C=C,
        max_iter=max_iter,
        tol=tol,
        loss=loss,
        fit_intercept=fit_intercept,
        early_stopping=early_stopping,
        random_state=42
    )

    # Cross-validation to evaluate performance
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(clf, X_train, y_train, cv=cv, scoring="roc_auc")
    
    return scores.mean()
'''

'\nimport optuna\nfrom sklearn.linear_model import PassiveAggressiveClassifier\nfrom sklearn.model_selection import cross_val_score\nfrom sklearn.metrics import roc_auc_score\nfrom sklearn.model_selection import StratifiedKFold\n\n\ndef objective(trial):\n    # Suggest hyperparameters\n    C = trial.suggest_loguniform("C", 1e-4, 1e2)   # regularization strength\n    max_iter = trial.suggest_int("max_iter", 500, 5000)\n    tol = trial.suggest_loguniform("tol", 1e-5, 1e-1)\n    loss = trial.suggest_categorical("loss", ["hinge", "squared_hinge"])\n    fit_intercept = trial.suggest_categorical("fit_intercept", [True, False])\n    early_stopping = trial.suggest_categorical("early_stopping", [True, False])\n    \n    # Define classifier\n    clf = PassiveAggressiveClassifier(\n        C=C,\n        max_iter=max_iter,\n        tol=tol,\n        loss=loss,\n        fit_intercept=fit_intercept,\n        early_stopping=early_stopping,\n        random_state=42\n    )\n\n    # Cross-validation to 

In [31]:
'''
study = optuna.create_study(direction="maximize")  # maximize ROC AUC
study.optimize(objective, n_trials=500)  # run 500 trials
'''

'\nstudy = optuna.create_study(direction="maximize")  # maximize ROC AUC\nstudy.optimize(objective, n_trials=500)  # run 500 trials\n'

In [32]:
'''
print("Best ROC AUC:", study.best_value)
print("Best hyperparameters:", study.best_params)

best_clf = PassiveAggressiveClassifier(**study.best_params, random_state=42)
best_clf.fit(X_train, y_train)

y_probabling = best_clf.decision_function(X_test)

auc_best = roc_auc_score(y_test, y_probabling)
print(auc_best)

clf_original = PassiveAggressiveClassifier(random_state=42)
clf_original.fit(X_train, y_train)
y_pr = clf_original.decision_function(X_test)

print(roc_auc_score(y_test, y_pr))
'''

'\nprint("Best ROC AUC:", study.best_value)\nprint("Best hyperparameters:", study.best_params)\n\nbest_clf = PassiveAggressiveClassifier(**study.best_params, random_state=42)\nbest_clf.fit(X_train, y_train)\n\ny_probabling = best_clf.decision_function(X_test)\n\nauc_best = roc_auc_score(y_test, y_probabling)\nprint(auc_best)\n\nclf_original = PassiveAggressiveClassifier(random_state=42)\nclf_original.fit(X_train, y_train)\ny_pr = clf_original.decision_function(X_test)\n\nprint(roc_auc_score(y_test, y_pr))\n'

In [33]:
#clf_original.get_params()

In [34]:
'''
from skopt import BayesSearchCV
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.model_selection import StratifiedKFold

# Define parameter search space
param_grid = {
    "C": (1e-4, 1e2, "log-uniform"),
    "max_iter": (500, 3000),
    "tol": (1e-5, 1e-1, "log-uniform"),
    "loss": ["hinge", "squared_hinge"],
    "fit_intercept": [True, False]
}

# Define classifier
pac = PassiveAggressiveClassifier(random_state=42)

# Use Bayesian Optimization
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
bayes_search = BayesSearchCV(
    pac,
    search_spaces=param_grid,
    n_iter=50,              # number of trials
    cv=cv,
    scoring="roc_auc",
    random_state=42,
    n_jobs=-1
)

# Run search
bayes_search.fit(X_train, y_train)

print("Best ROC AUC (CV):", bayes_search.best_score_)
print("Best parameters:", bayes_search.best_params_)

# Retrain best model
best_clf = bayes_search.best_estimator_
'''

'\nfrom skopt import BayesSearchCV\nfrom sklearn.linear_model import PassiveAggressiveClassifier\nfrom sklearn.model_selection import StratifiedKFold\n\n# Define parameter search space\nparam_grid = {\n    "C": (1e-4, 1e2, "log-uniform"),\n    "max_iter": (500, 3000),\n    "tol": (1e-5, 1e-1, "log-uniform"),\n    "loss": ["hinge", "squared_hinge"],\n    "fit_intercept": [True, False]\n}\n\n# Define classifier\npac = PassiveAggressiveClassifier(random_state=42)\n\n# Use Bayesian Optimization\ncv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)\nbayes_search = BayesSearchCV(\n    pac,\n    search_spaces=param_grid,\n    n_iter=50,              # number of trials\n    cv=cv,\n    scoring="roc_auc",\n    random_state=42,\n    n_jobs=-1\n)\n\n# Run search\nbayes_search.fit(X_train, y_train)\n\nprint("Best ROC AUC (CV):", bayes_search.best_score_)\nprint("Best parameters:", bayes_search.best_params_)\n\n# Retrain best model\nbest_clf = bayes_search.best_estimator_\n'

In [35]:
#best_clf.fit(X_train, y_train)

In [36]:
# Let's create a function that create the dataset 

def create_test_dataset(df: pd.DataFrame) -> pd.DataFrame:
    
    df = df.copy()

    # Apply build_prompt row by row
    prompts = df.apply(build_prompt, axis=1)

    # Build new dataframe
    new_df = pd.DataFrame({
        "prompts": prompts,
    })

    return new_df

test_dataset = create_test_dataset(test)
test_dataset

,prompts
0,\nYou are an expert content moderator. Your ta...
1,\nYou are an expert content moderator. Your ta...
2,\nYou are an expert content moderator. Your ta...
3,\nYou are an expert content moderator. Your ta...
4,\nYou are an expert content moderator. Your ta...
5,\nYou are an expert content moderator. Your ta...
6,\nYou are an expert content moderator. Your ta...
7,\nYou are an expert content moderator. Your ta...
8,\nYou are an expert content moderator. Your ta...
9,\nYou are an expert content moderator. Your ta...


In [37]:
X_test_sub = get_embeddings(test_dataset["prompts"].tolist())

X_test_sub

Embedding texts: 100%|██████████| 1/1 [00:00<00:00,  1.42batch/s]


array([[ 1.7057474 , -1.7262087 , -1.1607841 , ...,  1.5098134 ,
         1.986594  ,  2.0328588 ],
       [ 0.5457788 , -0.4181694 , -0.7969366 , ..., -0.8237892 ,
         0.10895566, -0.6196605 ],
       [-0.10764619, -2.7696846 , -1.064669  , ...,  0.64693505,
        -0.03715123,  0.8456739 ],
       ...,
       [ 0.7920315 , -2.6101375 , -1.2220678 , ...,  2.060607  ,
         1.9341587 ,  1.9223617 ],
       [-0.5422405 , -2.780406  , -1.1153296 , ...,  1.3238841 ,
         2.1037245 ,  2.2305138 ],
       [ 0.26227617, -1.9449751 , -1.0905702 , ...,  1.8414946 ,
         1.6508467 ,  1.1829212 ]], dtype=float32)

In [38]:
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.calibration import CalibratedClassifierCV

pac = PassiveAggressiveClassifier(random_state=42)
clf_original = CalibratedClassifierCV(pac, method="sigmoid", cv=5)  # Platt scaling
clf_original.fit(X_train, y_train)

sub = clf_original.predict_proba(X_test_sub)

sub

array([[0.6973902 , 0.3026098 ],
       [0.57410689, 0.42589311],
       [0.43652785, 0.56347215],
       [0.59011751, 0.40988249],
       [0.29989827, 0.70010173],
       [0.59376953, 0.40623047],
       [0.28813215, 0.71186785],
       [0.75619708, 0.24380292],
       [0.67439836, 0.32560164],
       [0.28795366, 0.71204634]])

In [39]:
submission = pd.DataFrame({'row_id' : test.row_id, 'rule_violation' : sub[:,1]})

In [40]:
submission.to_csv('/kaggle/working/submission.csv')

# Let's start making the Dataset (to train the model using qwen)

In [41]:
#pip install torch transformers datasets peft optuna scikit-learn tqdm cleantext

In [42]:
'''
import torch
print(torch.cuda.is_available())
'''

'\nimport torch\nprint(torch.cuda.is_available())\n'

In [43]:
'''
# --------- 1) Load your CSV into HF Dataset (you already have df)
train_dataset_shuffle = train_dataset.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

# Map labels (already 0/1)
dataset = Dataset.from_pandas(train_dataset_shuffle)
'''

'\n# --------- 1) Load your CSV into HF Dataset (you already have df)\ntrain_dataset_shuffle = train_dataset.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle\n\n# Map labels (already 0/1)\ndataset = Dataset.from_pandas(train_dataset_shuffle)\n'

In [44]:
#dataset

In [45]:
'''
# --------- 2) Tokenizer & tokenization
MODEL_NAME = "/kaggle/input/qwen-3-embedding/transformers/0.6b/1"  # pick appropriate Qwen-3 variant or a smaller one to test
tokenizer = AutoTokenizer.from_pretrained(model, trust_remote_code=True)

MAX_LEN = 512
def tokenize_fn(examples):
    return tokenizer(examples["prompts"], truncation=True, padding=False, max_length=MAX_LEN)
'''

'\n# --------- 2) Tokenizer & tokenization\nMODEL_NAME = "/kaggle/input/qwen-3-embedding/transformers/0.6b/1"  # pick appropriate Qwen-3 variant or a smaller one to test\ntokenizer = AutoTokenizer.from_pretrained(model, trust_remote_code=True)\n\nMAX_LEN = 512\ndef tokenize_fn(examples):\n    return tokenizer(examples["prompts"], truncation=True, padding=False, max_length=MAX_LEN)\n'

In [46]:
'''
# We will tokenise per split below to avoid leakage across folds

# --------- Utility: compute_metrics for Trainer using ROC-AUC
def compute_metrics(pred):
    logits = pred.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    # For binary classification HF often returns shape (N,2); use softmax prob for class 1
    if logits.shape[1] == 2:
        probs = 1.0 / (1.0 + np.exp(- (logits[:,1])))  # actually softmax, but easier to get prob for class 1:
        # Better: softmax -> probs[:,1]
        from scipy.special import softmax
        probs = softmax(logits, axis=1)[:,1]
    else:
        # single logit -> sigmoid
        probs = 1.0 / (1.0 + np.exp(-logits.ravel()))
    labels = pred.label_ids
    try:
        auc = roc_auc_score(labels, probs)
    except ValueError:
        auc = float("nan")
    return {"roc_auc": auc}
'''

'\n# We will tokenise per split below to avoid leakage across folds\n\n# --------- Utility: compute_metrics for Trainer using ROC-AUC\ndef compute_metrics(pred):\n    logits = pred.predictions\n    if isinstance(logits, tuple):\n        logits = logits[0]\n    # For binary classification HF often returns shape (N,2); use softmax prob for class 1\n    if logits.shape[1] == 2:\n        probs = 1.0 / (1.0 + np.exp(- (logits[:,1])))  # actually softmax, but easier to get prob for class 1:\n        # Better: softmax -> probs[:,1]\n        from scipy.special import softmax\n        probs = softmax(logits, axis=1)[:,1]\n    else:\n        # single logit -> sigmoid\n        probs = 1.0 / (1.0 + np.exp(-logits.ravel()))\n    labels = pred.label_ids\n    try:\n        auc = roc_auc_score(labels, probs)\n    except ValueError:\n        auc = float("nan")\n    return {"roc_auc": auc}\n'